In [11]:
import math

import sqlite3
import numpy as np
import matplotlib.pyplot as plt

In [3]:
connection = sqlite3.connect('player_batting.db')
cursor     = connection.cursor()

playerIds = []

playerIdQuery = "SELECT DISTINCT player_id FROM aggregate_weekly_grades"

cursor.execute(playerIdQuery)

rows = cursor.fetchall()

for row in rows:
    playerId = row[0]

    playerIds.append(playerId)

cursor    .close()
connection.close()

print(len(playerIds))

651


In [14]:
def get_r_value(xValues, yValues):
    xAvg = sum(xValues) / len(xValues)
    yAvg = sum(yValues) / len(yValues)

    diffSum         = 0.0
    squaredDiffSumX = 0.0
    squaredDiffSumY = 0.0

    for i in range(len(xValues)):
        curX = xValues[i]
        curY = yValues[i]

        diffX = curX - xAvg
        diffY = curY - yAvg

        diffSum = diffSum + diffX * diffY

        squaredDiffSumX = squaredDiffSumX + diffX**2
        squaredDiffSumY = squaredDiffSumY + diffY**2

    totalSquaredDiffSum = math.sqrt(squaredDiffSumX * squaredDiffSumY)

    if totalSquaredDiffSum == 0.0:
        return 0.0

    return diffSum / totalSquaredDiffSum


In [37]:
playerRunsByWeek        = {}
playerHomeRunsByWeek    = {}
playerRbisByWeek        = {}
playerStolenBasesByWeek = {}
playerObpByWeek         = {}
playerTotalByWeek       = {}

playerRunsRvalueByWeek        = {}
playerHomeRunsRvalueByWeek    = {}
playerRbisRvalueByWeek        = {}
playerStolenBasesRvalueByWeek = {}
playerObpRvalueByWeek         = {}
playerTotalRvalueByWeek       = {}

playerRunsExtendedRvalueByWeek        = {}
playerHomeRunsExtendedRvalueByWeek    = {}
playerRbisExtendedRvalueByWeek        = {}
playerStolenBasesExtendedRvalueByWeek = {}
playerObpExtendedRvalueByWeek         = {}
playerTotalExtendedRvalueByWeek       = {}

for i in range(1,30):
    playerRunsByWeek       [i] = {}
    playerHomeRunsByWeek   [i] = {}
    playerRbisByWeek       [i] = {}
    playerStolenBasesByWeek[i] = {}
    playerObpByWeek        [i] = {}
    playerTotalByWeek      [i] = {}

    playerRunsRvalueByWeek       [i] = {} 
    playerHomeRunsRvalueByWeek   [i] = {} 
    playerRbisRvalueByWeek       [i] = {} 
    playerStolenBasesRvalueByWeek[i] = {} 
    playerObpRvalueByWeek        [i] = {} 
    playerTotalRvalueByWeek      [i] = {}

    playerRunsExtendedRvalueByWeek       [i] = {}
    playerHomeRunsExtendedRvalueByWeek   [i] = {}
    playerRbisExtendedRvalueByWeek       [i] = {}
    playerStolenBasesExtendedRvalueByWeek[i] = {}
    playerObpExtendedRvalueByWeek        [i] = {}
    playerTotalExtendedRvalueByWeek      [i] = {}
    
    for playerId in playerIds:
        query = f"SELECT stat,num,percentile FROM weekly_grades WHERE player_id = \"{playerId}\" AND week_number = {i}"
        
        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(query)

        rows = cursor.fetchall()

        seenStats = set()

        for row in rows:
            stat       = row[0]
            num        = float(row[1])
            percentile = float(row[2])

            seenStats.add(stat)

            if stat == "RUN":
                playerRunsByWeek[i][playerId] = num
            elif stat == "HOME_RUN":
                playerHomeRunsByWeek[i][playerId] = num
            elif stat == "RBI":
                playerRbisByWeek[i][playerId] = num
            elif stat == "STOLEN_BASE":
                playerStolenBasesByWeek[i][playerId] = num
            elif stat == "ON_BASE_PERCENTAGE":
                playerObpByWeek[i][playerId] = num
            elif stat == "TOTAL":
                playerTotalByWeek[i][playerId] = num

        cursor    .close()
        connection.close()

        if "RUN" not in seenStats:
            playerRunsByWeek[i][playerId] = 0.0
        if "HOME_RUN" not in seenStats:
            playerHomeRunsByWeek[i][playerId] = 0.0
        if "RBI" not in seenStats:
            playerRbisByWeek[i][playerId] = 0.0
        if "STOLEN_BASE" not in seenStats:
            playerStolenBasesByWeek[i][playerId] = 0.0
        if "ON_BASE_PERCENTAGE" not in seenStats:
            playerObpByWeek[i][playerId] = 0.0
        if "TOTAL" not in seenStats:
            playerTotalByWeek[i][playerId] = 0.0

        if i == 1:
            continue
        if i >= 2:
            curWeekRuns      = playerRunsByWeek[i    ][playerId]
            previousWeekRuns = playerRunsByWeek[i - 1][playerId]

            rValueRuns = get_r_value([previousWeekRuns, curWeekRuns], [i - 1, i])

            curWeekHomeRuns      = playerHomeRunsByWeek[i    ][playerId]
            previousWeekHomeRuns = playerHomeRunsByWeek[i - 1][playerId]

            rValueHomeRuns = get_r_value([previousWeekHomeRuns, curWeekHomeRuns], [i - 1, i])

            curWeekRbis      = playerRbisByWeek[i    ][playerId]
            previousWeekRbis = playerRbisByWeek[i - 1][playerId]

            rValueRbis = get_r_value([previousWeekRbis, curWeekRbis], [i - 1, i])

            curWeekSbs      = playerStolenBasesByWeek[i    ][playerId]
            previousWeekSbs = playerStolenBasesByWeek[i - 1][playerId]

            rValueSbs = get_r_value([previousWeekSbs, curWeekSbs], [i - 1, i])

            curWeekObs      = playerTotalByWeek[i    ][playerId]
            previousWeekObs = playerTotalByWeek[i - 1][playerId]

            rValueObs = get_r_value([previousWeekObs, curWeekObs], [i - 1, i])

            curWeekTotal      = playerTotalByWeek[i    ][playerId]
            previousWeekTotal = playerTotalByWeek[i - 1][playerId]

            rValueTotal = get_r_value([previousWeekTotal, curWeekTotal], [i - 1, i])

            playerRunsRvalueByWeek       [i][playerId] = rValueRuns 
            playerHomeRunsRvalueByWeek   [i][playerId] = rValueHomeRuns 
            playerRbisRvalueByWeek       [i][playerId] = rValueRbis 
            playerStolenBasesRvalueByWeek[i][playerId] = rValueSbs 
            playerObpRvalueByWeek        [i][playerId] = rValueObs 
            playerTotalRvalueByWeek      [i][playerId] = rValueTotal

            if playerId == "ralec001":
                print(f"{i}, {rValueHomeRuns}")
        if i >= 3:
            curWeekRuns      = playerRunsByWeek[i    ][playerId]
            previousWeekRuns = playerRunsByWeek[i - 1][playerId]
            twoWeekRuns      = playerRunsByWeek[i - 2][playerId]

            rValueRuns = get_r_value([twoWeekRuns, previousWeekRuns, curWeekRuns], [i - 2, i - 1, i])

            curWeekHomeRuns      = playerHomeRunsByWeek[i    ][playerId]
            previousWeekHomeRuns = playerHomeRunsByWeek[i - 1][playerId]
            twoWeekHomeRuns      = playerHomeRunsByWeek[i - 2][playerId]

            rValueHomeRuns = get_r_value([twoWeekHomeRuns, previousWeekHomeRuns, curWeekHomeRuns], [i - 2, i - 1, i])

            curWeekRbis      = playerRbisByWeek[i    ][playerId]
            previousWeekRbis = playerRbisByWeek[i - 1][playerId]
            twoWeekRbis      = playerRbisByWeek[i - 2][playerId]

            rValueRbis = get_r_value([twoWeekRbis, previousWeekRbis, curWeekRbis], [i - 2, i - 1, i])

            curWeekSbs      = playerStolenBasesByWeek[i    ][playerId]
            previousWeekSbs = playerStolenBasesByWeek[i - 1][playerId]
            twoWeekSbs      = playerStolenBasesByWeek[i - 2][playerId]

            rValueSbs = get_r_value([twoWeekSbs, previousWeekSbs, curWeekSbs], [i - 2, i - 1, i])

            curWeekObs      = playerTotalByWeek[i    ][playerId]
            previousWeekObs = playerTotalByWeek[i - 1][playerId]
            twoWeekObs      = playerTotalByWeek[i - 2][playerId]

            rValueObs = get_r_value([twoWeekObs, previousWeekObs, curWeekObs], [i - 2, i - 1, i])

            curWeekTotal      = playerTotalByWeek[i    ][playerId]
            previousWeekTotal = playerTotalByWeek[i - 1][playerId]
            twoWeekTotal      = playerTotalByWeek[i - 2][playerId]

            rValueTotal = get_r_value([twoWeekTotal, previousWeekTotal, curWeekTotal], [i - 2, i - 1, i])

            playerRunsExtendedRvalueByWeek       [i][playerId] = rValueRuns 
            playerHomeRunsExtendedRvalueByWeek   [i][playerId] = rValueHomeRuns 
            playerRbisExtendedRvalueByWeek       [i][playerId] = rValueRbis 
            playerStolenBasesExtendedRvalueByWeek[i][playerId] = rValueSbs 
            playerObpExtendedRvalueByWeek        [i][playerId] = rValueObs 
            playerTotalExtendedRvalueByWeek      [i][playerId] = rValueTotal
            

2, 0.0
3, 0.0
4, 1.0
5, 1.0
6, -1.0
7, 1.0
8, 0.0
9, -1.0
10, 1.0
11, -1.0
12, 0.0
13, 1.0
14, -1.0
15, 1.0
16, -1.0
17, 1.0
18, -1.0
19, 1.0
20, -1.0
21, 1.0
22, -1.0
23, -1.0
24, 0.0
25, 1.0
26, -1.0
27, 0.0
28, 1.0
29, -1.0


In [56]:
print(playerObpExtendedRvalueByWeek[20]["ralec001"])

0.6063686194072255


In [94]:
playerRunsByWeek        = {}
playerHomeRunsByWeek    = {}
playerRbisByWeek        = {}
playerStolenBasesByWeek = {}
playerObpByWeek         = {}
playerTotalByWeek       = {}

playerRunsRankByWeek        = {}
playerHomeRunsRankByWeek    = {}
playerRbisRankByWeek        = {}
playerStolenBasesRankByWeek = {}
playerObpRankByWeek         = {}
playerTotalRankByWeek       = {}

playerRunsDifferenceByWeek        = {}
playerHomeRunsDifferenceByWeek    = {}
playerRbisDifferenceByWeek        = {}
playerStolenBasesDifferenceByWeek = {}
playerObpDifferenceByWeek         = {}
playerTotalDifferenceByWeek       = {}

for i in range(1,30):
    playerRunsByWeek       [i] = {}
    playerHomeRunsByWeek   [i] = {}
    playerRbisByWeek       [i] = {}
    playerStolenBasesByWeek[i] = {}
    playerObpByWeek        [i] = {}
    playerTotalByWeek      [i] = {}

    playerRunsDifferenceByWeek       [i] = {} 
    playerHomeRunsDifferenceByWeek   [i] = {} 
    playerRbisDifferenceByWeek       [i] = {} 
    playerStolenBasesDifferenceByWeek[i] = {} 
    playerObpDifferenceByWeek        [i] = {} 
    playerTotalDifferenceByWeek      [i] = {}

    playerRunsRankByWeek        [i] = {}
    playerHomeRunsRankByWeek    [i] = {}
    playerRbisRankByWeek        [i] = {}
    playerStolenBasesRankByWeek [i] = {}
    playerObpRankByWeek         [i] = {}
    playerTotalRankByWeek       [i] = {}

    playerToRuns        = []
    playerToHomeRuns    = []
    playerToRbis        = []
    playerToStolenBases = []
    playerToObp         = []
    playerToTotal       = []
    
    for playerId in playerIds:
        query = f"SELECT stat,num,percentile FROM weekly_grades WHERE player_id = \"{playerId}\" AND week_number = {i}"
        
        connection = sqlite3.connect('player_batting.db')
        cursor     = connection.cursor()

        cursor.execute(query)

        rows = cursor.fetchall()

        seenStats = set()

        for row in rows:
            stat       = row[0]
            num        = float(row[1])
            percentile = float(row[2])

            seenStats.add(stat)

            if stat == "RUN":
                playerRunsByWeek[i][playerId] = num
                playerToRuns.append((playerId, num))
            elif stat == "HOME_RUN":
                playerHomeRunsByWeek[i][playerId] = num
                playerToHomeRuns.append((playerId, num))
            elif stat == "RBI":
                playerRbisByWeek[i][playerId] = num
                playerToRbis.append((playerId, num))
            elif stat == "STOLEN_BASE":
                playerStolenBasesByWeek[i][playerId] = num
                playerToStolenBases.append((playerId, num))
            elif stat == "ON_BASE_PERCENTAGE":
                playerObpByWeek[i][playerId] = num
                playerToObp.append((playerId, num))
            elif stat == "TOTAL":
                playerTotalByWeek[i][playerId] = num
                playerToTotal.append((playerId, num))

        cursor    .close()
        connection.close()

        if "RUN" not in seenStats:
            playerRunsByWeek[i][playerId] = 0.0
            playerToRuns.append((playerId, 0.0))
        if "HOME_RUN" not in seenStats:
            playerHomeRunsByWeek[i][playerId] = 0.0
            playerToHomeRuns.append((playerId, 0.0))
        if "RBI" not in seenStats:
            playerRbisByWeek[i][playerId] = 0.0
            playerToRbis.append((playerId, 0.0))
        if "STOLEN_BASE" not in seenStats:
            playerStolenBasesByWeek[i][playerId] = 0.0
            playerToStolenBases.append((playerId, 0.0))
        if "ON_BASE_PERCENTAGE" not in seenStats:
            playerObpByWeek[i][playerId] = 0.0
            playerToObp.append((playerId, 0.0))
        if "TOTAL" not in seenStats:
            playerTotalByWeek[i][playerId] = 0.0
            playerToTotal.append((playerId, 0.0))

        if i == 1:
            continue
        if i >= 2:
            curWeekRuns      = playerRunsByWeek[i    ][playerId]
            previousWeekRuns = playerRunsByWeek[i - 1][playerId]

            differenceRuns = curWeekRuns - previousWeekRuns

            curWeekHomeRuns      = playerHomeRunsByWeek[i    ][playerId]
            previousWeekHomeRuns = playerHomeRunsByWeek[i - 1][playerId]

            differenceHomeRuns = curWeekHomeRuns - previousWeekHomeRuns

            curWeekRbis      = playerRbisByWeek[i    ][playerId]
            previousWeekRbis = playerRbisByWeek[i - 1][playerId]

            differenceRbis = curWeekRbis - previousWeekRbis

            curWeekSbs      = playerStolenBasesByWeek[i    ][playerId]
            previousWeekSbs = playerStolenBasesByWeek[i - 1][playerId]

            differenceSbs = curWeekSbs - previousWeekSbs

            curWeekObp      = playerObpByWeek[i    ][playerId]
            previousWeekObp = playerObpByWeek[i - 1][playerId]

            differenceObp = curWeekObp - previousWeekObp

            curWeekTotal      = playerTotalByWeek[i    ][playerId]
            previousWeekTotal = playerTotalByWeek[i - 1][playerId]

            differenceTotal = curWeekTotal - previousWeekTotal

            #UPDATE weekly_grades SET week_change = differenceRuns WHERE player_id = playerId AND week = i AND stat = "RUN"
            #UPDATE weekly_grades SET week_change = differenceHomeRuns WHERE player_id = playerId AND week = i AND stat = "HOME_RUN"
            #UPDATE weekly_grades SET week_change = differenceRbis WHERE player_id = playerId AND week = i AND stat = "RBI"
            #UPDATE weekly_grades SET week_change = differenceSbs WHERE player_id = playerId AND week = i AND stat = "STOLEN_BASE"
            #UPDATE weekly_grades SET week_change = differenceObp WHERE player_id = playerId AND week = i AND stat = "ON_BASE_PERCENTAGE"
            #UPDATE weekly_grades SET week_change = differenceTotal WHERE player_id = playerId AND week = i AND stat = "TOTAL"

            playerRunsDifferenceByWeek       [i][playerId] = differenceRuns 
            playerHomeRunsDifferenceByWeek   [i][playerId] = differenceHomeRuns 
            playerRbisDifferenceByWeek       [i][playerId] = differenceRbis 
            playerStolenBasesDifferenceByWeek[i][playerId] = differenceSbs 
            playerObpDifferenceByWeek        [i][playerId] = differenceObp 
            playerTotalDifferenceByWeek      [i][playerId] = differenceTotal

            if playerId == "ralec001":
                print(f"{i}, {differenceHomeRuns}")     

    playerToRuns        .sort(key=lambda x: x[1], reverse=True)
    playerToHomeRuns    .sort(key=lambda x: x[1], reverse=True)
    playerToRbis        .sort(key=lambda x: x[1], reverse=True)
    playerToStolenBases .sort(key=lambda x: x[1], reverse=True)
    playerToObp         .sort(key=lambda x: x[1], reverse=True)
    playerToTotal       .sort(key=lambda x: x[1], reverse=True)

    playerRunsRankByWeek        [i] = {}
    playerHomeRunsRankByWeek    [i] = {}
    playerRbisRankByWeek        [i] = {}
    playerStolenBasesRankByWeek [i] = {}
    playerObpRankByWeek         [i] = {}
    playerTotalRankByWeek       [i] = {}

    rank     = 1
    previous = playerToRuns[0][1]

    #ONLY INCREMENT RANK WHEN THE CURRENT VALUE IS NOT EQUAL TO THE PREVIOUS
    for playerToRun in playerToRuns:
        if previous != playerToRun[1]:
            rank = rank + 1

        previous = playerToRun[1]
        
        playerRunsRankByWeek[i][playerToRun[0]] = rank

    rank     = 1
    previous = playerToHomeRuns[0][1]
    
    for playerToHomeRun in playerToHomeRuns:
        if previous != playerToHomeRun[1]:
            rank = rank + 1

        previous = playerToHomeRun[1]
       
        playerHomeRunsRankByWeek[i][playerToHomeRun[0]] = rank

    rank     = 1
    previous = playerToRbis[0][1]

    for playerToRbi in playerToRbis:
        if previous != playerToRbi[1]:
            rank = rank + 1

        previous = playerToRbi[1]
        
        playerRbisRankByWeek[i][playerToRbi[0]] = rank

    rank     = 1
    previous = playerToStolenBases[0][1]

    for playerToStolenBase in playerToStolenBases:
        if previous != playerToStolenBase[1]:
            rank = rank + 1

        previous = playerToStolenBase[1]
            
        playerStolenBasesRankByWeek[i][playerToStolenBase[0]] = rank

    rank     = 1
    previous = playerToObp[0][1]
    
    for playerObp in playerToObp:
        if previous != playerObp[1]:
            rank = rank + 1

        previous = playerObp[1]
            
        playerObpRankByWeek[i][playerObp[0]] = rank

    rank     = 1
    previous = playerToTotal[0][1]

    for playerTotal in playerToTotal:
        if previous != playerTotal[1]:
            rank = rank + 1

        previous = playerTotal[1]
            
        playerTotalRankByWeek[i][playerTotal[0]] = rank        

2, 0.0
3, 0.0
4, 2.0
5, 1.0
6, -2.0
7, 1.0
8, 0.0
9, -2.0
10, 1.0
11, -1.0
12, 0.0
13, 1.0
14, -1.0
15, 2.0
16, -1.0
17, 4.0
18, -5.0
19, 2.0
20, -1.0
21, 2.0
22, -2.0
23, -1.0
24, 0.0
25, 2.0
26, -1.0
27, 0.0
28, 2.0
29, -3.0


In [104]:
print(playerTotalDifferenceByWeek[2]["ohtas001"])

-1.2329920089081026


In [ ]:
playerRunsDifferenceByWeek       
playerHomeRunsDifferenceByWeek   
playerRbisDifferenceByWeek       
playerStolenBasesDifferenceByWeek
playerObpDifferenceByWeek        
playerTotalDifferenceByWeek      